In [ ]:
from datetime import date

import polars as pl

from flights.evaluation.evaluation import main

test_date = date(2026, 3, 1)
results, df_stats = main(test_src="opensky", gold_src="sfdps", test_dates=test_date)
df = results[0]

In [ ]:
COLUMNS = [
    "icao", "takeoff_time", "landing_time",
    "takeoff_airport_ident", "landing_airport_ident",
    "takeoff_time_df1", "landing_time_df1",
    "takeoff_airport_ident_df1", "landing_airport_ident_df1",
    "match_status", "same_takeoff_airport_ident",
    "same_landing_airport_ident", "same_airport_ident",
]
mismatches = df.filter(pl.col("match_status") != "both").select(COLUMNS)
mismatches

In [ ]:
mismatch_summary = (
    mismatches.group_by("icao")
    .agg(
        (pl.col("match_status") == "df0_only").sum().alias("df0_only"),
        (pl.col("match_status") == "df1_only").sum().alias("df1_only"),
    )
    .sort("icao")
)

confirmed_gold_error_icaos = ["a1e6b9", "a3a1dc", "a6ea99"]
confirmed_both_error_icaos = [
    "a1a4a7", "a1a7ad", "a1a7dd", "a1c2e3", "a1d684",
    "a25dca", "a3e348", "a5a744", "a63c0b", "a654ec",
    "a84a06", "a875ab", "abfc78", "ac97c2", "ad963b",
    "adccd0",
]

mismatch_classification = mismatch_summary.with_columns(
    pl.when(pl.col("icao").is_in(confirmed_both_error_icaos))
    .then(pl.lit("both"))
    .when(pl.col("icao").is_in(confirmed_gold_error_icaos))
    .then(pl.lit("df1_gold_incorrect"))
    .otherwise(pl.lit("opensky_source_incorrect"))
    .alias("classification")
)
mismatch_classification

In [ ]:
# Airport-only failures are not included in match_status mismatches because
# this evaluation intentionally runs with compare_airports=False.
airport_only_errors = df.filter(
    (pl.col("match_status") == "both") & ~pl.col("same_airport_ident")
).select(COLUMNS)
airport_only_errors

## Manual review of direct OpenSky flight mismatches

This is the direct OpenSky flight source (`get_flights(..., algorithm="opensky")`), not the PlaneQuery flight algorithm using OpenSky state vectors.

### Summary

- **6,041 true positives, 219 false positives, and 689 false negatives**
- **588 mismatched ICAOs / 908 mismatched rows**
- **Confirmed df1/gold errors:** 3 ICAOs
- **Both sources incorrect:** 16 ICAOs
- **Direct OpenSky source failures:** 569 ICAOs
- Only **49.33%** of time-matched flights have both endpoint airport identifiers correct.

### Mismatch error percentages

Percentages use the 588 mismatched ICAOs as the denominator.

| Test set Error (%) | Gold-Dataset Error (%) | Error in both (%) |
|---:|---:|---:|
| 96.77% | 0.51% | 2.72% |

The direct OpenSky failures are primarily missing flights, partial tracks, merged multi-leg tracks, and incorrect or null endpoint airports. The `mismatch_classification` dataframe contains every time-mismatched ICAO. The separate `airport_only_errors` dataframe is essential because `compare_airports=False` hides airport-only failures from `match_status`.

### Confirmed df1/gold errors

| ICAO | Reason |
|---|---|
| `a1e6b9` | KHOU-KDAL actually departs around 13:39; df1 says 12:34. |
| `a3a1dc` | KRBD-KMKY actually departs around 21:10; df1 reuses 18:11. |
| `a6ea99` | KGYY-KLNK actually departs around 16:37; df1 says 15:30. |

### Both OpenSky and df1 are incorrect

| ICAOs | Explanation |
|---|---|
| `a1a4a7`, `a1a7ad`, `a1a7dd`, `a1c2e3`, `a1d684`, `a25dca`, `a3e348`, `a5a744`, `a63c0b`, `a654ec`, `a84a06`, `a875ab`, `abfc78`, `ac97c2`, `ad963b`, `adccd0` | df1 has a known combined-leg, omitted-leg, or timestamp error, while OpenSky also misses part of the trajectory or assigns incorrect/null airport endpoints. |

### OpenSky-only failure patterns

All remaining 569 time-mismatched ICAOs are attributed to the direct OpenSky source. Common patterns are:

- a valid SFDPS flight is entirely absent;
- only the beginning or end of the track is present;
- multiple real legs are merged into one long flight;
- coverage gaps create false short flights;
- takeoff or landing airport identifiers are null or assigned to unrelated nearby airports.

> **Comparison caveat:** the 588-ICAO count only covers time-based `df0_only`/`df1_only` mismatches. Airport-only errors among `both` rows are substantially more numerous and are shown separately above.

In [ ]:
# Deterministic smallest mismatch for a focused trajectory investigation.
unmatched_icao = (
    mismatches.group_by("icao")
    .agg(pl.len().alias("flight_count"))
    .sort(["flight_count", "icao"])
    .get_column("icao")
    .item(0)
)
unmatched_icao